In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split
import os

In [ ]:
#  FILE PATH
csv_path = os.path.join('..', 'data', 'emotify_final_dataset.csv')
base_folder = os.path.join('..', 'data', 'Emotify_npy') 


def load_dataset(csv_path, base_folder='data', threshold=0.4):
    df = pd.read_csv(csv_path)
    X = []
    emotion_cols = ['amazement', 'solemnity', 'tenderness', 'nostalgia', 'calmness', 
                    'power', 'joyful_activation', 'tension', 'sadness']
    
    # Threshold
    y = (df[emotion_cols].values >= threshold).astype(int)

    print(f"Loading {len(df)} files with threshold {threshold}...")
    for path in df['npy_path']:
        full_path = os.path.join(base_folder, path)
        data = np.load(full_path)
        # Normalization
        data_min, data_max = data.min(), data.max()
        if data_max - data_min != 0:
            data = (data - data_min) / (data_max - data_min)
        X.append(data)
    
    X = np.array(X)
    X = np.expand_dims(X, axis=-1) 
    return X, y, emotion_cols

crnn #1 trial

In [ ]:
# CRNN Model optimization
def build_crnn_model(input_shape=(128, 1292, 1)):
    model = models.Sequential([
        # CNN Layers: local feature extraction
        layers.Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=input_shape),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        
        layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        
        layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),

        # Reshape for RNN
        # (16, 161, 64) -> (161, 1024)
        layers.Reshape(target_shape=(161, 16 * 64)), 

        # RNN Layer (GRU): time-series feature extraction
        layers.GRU(64, return_sequences=False),
        layers.Dropout(0.3), # Regularization fighting overfitting

        layers.Dense(64, activation='relu'),
        layers.Dense(9, activation='sigmoid') # Multi-label EXIT
    ])
    
    # Χρήση σταθερού, προσεκτικού learning rate
    optimizer = tf.keras.optimizers.Adam(learning_rate=0.0005)
    model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])
    return model

In [ ]:


# new threshold
X, y, labels = load_dataset(csv_path, base_folder, threshold=0.4)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# CRNN
model = build_crnn_model(input_shape=X_train.shape[1:])

# Callback
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss', 
    patience=10, 
    restore_best_weights=True
)

#Learning
history = model.fit(
    X_train, y_train,
    epochs=50,
    batch_size=16,
    validation_data=(X_test, y_test),
    callbacks=[early_stop],
    verbose=1
)

In [ ]:
import matplotlib.pyplot as plt


plt.figure(figsize=(15, 5))

# Accuracy
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Training', linewidth=2)
plt.plot(history.history['val_accuracy'], label='Validation', linewidth=2)
plt.title('Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend(loc='lower right')
plt.grid(True)

#  Loss 
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Training', linewidth=2)
plt.plot(history.history['val_loss'], label='Validation', linewidth=2)
plt.title('Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend(loc='upper right')
plt.grid(True)

plt.tight_layout()
plt.show()

Prominent Overfir

Aggregation - Less labels

In [ ]:
#new dataset loader with threshold
def load_dataset(csv_path, base_folder='data', threshold=0.4):
    df = pd.read_csv(csv_path)
    X = []

    # new labels 4 from 9
    emotion_cols = ['energy_joy', 'calm_relax', 'sadness_nostalgia', 'tension_fear']
    
    # values from last cell
    y = df[emotion_cols].values

    print(f"Loading {len(df)} files from {csv_path}...")
    
  
    for path in df['file_path']:
       
        full_path = os.path.join(base_folder, path)
        npy_path = full_path.replace('Emotify_Audio', 'Emotify_npy').replace('.mp3', '.npy')
        
        if not os.path.exists(npy_path):
            print(f"Error: The file was not found at: {npy_path}")
            continue
            
        data = np.load(npy_path)
        
        #Normalization (Min-Max)
        data_min, data_max = data.min(), data.max()
        if data_max - data_min != 0:
            data = (data - data_min) / (data_max - data_min)
        X.append(data)
    
    X = np.array(X)
    X = np.expand_dims(X, axis=-1) 
    return X, y, emotion_cols

Larger Dataset: ADD noise

In [ ]:
def add_noise(data, factor=0.005):
    noise = np.random.randn(*data.shape)
    return data + factor * noise

def load_dataset(csv_path, base_folder='data'):
    df = pd.read_csv(csv_path)
    X = []
    Y = []
    
    emotion_cols = ['energy_joy', 'calm_relax', 'sadness_nostalgia', 'tension_fear']
    
    print(f"Loading and Augmenting 400 files from {csv_path}...")
    
    for i, path in enumerate(df['file_path']):

        full_path = os.path.join(base_folder, path)
        npy_path = full_path.replace('Emotify_Audio', 'Emotify_npy').replace('.mp3', '.npy')
        
        if not os.path.exists(npy_path):
            continue
            
        #load the npy file
        data = np.load(npy_path)
        data_min, data_max = data.min(), data.max()
        if data_max - data_min != 0:
            data = (data - data_min) / (data_max - data_min)
        
        # protoype: X.append(data)
        X.append(data)
        Y.append(df.iloc[i][emotion_cols].values) # label
        
        # add noise
        data_noisy = add_noise(data)
        X.append(data_noisy)
        Y.append(df.iloc[i][emotion_cols].values) # same label 
        
    X = np.array(X)
    X = np.expand_dims(X, axis=-1)
    y = np.array(Y).astype(float) 
    
    print(f"Final dataset size: {len(X)} samples (doubled!)")
    return X, y, emotion_cols

CRNN - trial #2

In [ ]:
#CRNN optimization
def build_crnn_model(input_shape=(128, 1292, 1)):
    model = models.Sequential([

        layers.Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=input_shape),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        
        layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        
        layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),

        # Reshape for RNN
        # (16, 161, 64) -> (161, 1024)
        layers.Reshape(target_shape=(161, 16 * 64)), 

        #RNN Layer (GRU): time-series feature extraction
        layers.GRU(64, return_sequences=False),
        layers.Dropout(0.3), # Regularization 

        layers.Dense(64, activation='relu'),
        layers.Dense(4, activation='sigmoid') # Multi-label exit
    ])
    
    #stable learning rate
    optimizer = tf.keras.optimizers.Adam(learning_rate=0.0005)
    model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])
    return model

In [ ]:

#  FILE PATH
csv_path = os.path.join('..', 'data', 'emotify_final_dataset.csv')
base_folder = os.path.join('..', 'data', 'Emotify_npy') 


X, y, labels = load_dataset(csv_path, base_folder)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# CRNN with 4 outputs
model = build_crnn_model(input_shape=X_train.shape[1:])

#Learning
history = model.fit(
    X_train, y_train,
    epochs=50,
    batch_size=16,
    validation_data=(X_test, y_test),
    callbacks=[early_stop],
    verbose=1
)

In [ ]:
import matplotlib.pyplot as plt


plt.figure(figsize=(15, 5))
# Accuracy 
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Training', linewidth=2)
plt.plot(history.history['val_accuracy'], label='Validation', linewidth=2)
plt.title('Model Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend(loc='lower right')
plt.grid(True)

# Loss 
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Training', linewidth=2)
plt.plot(history.history['val_loss'], label='Validation', linewidth=2)
plt.title('Model Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend(loc='upper right')
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

y_pred = (model.predict(X_test) > 0.7).astype(int) #0.5 arxiko
# for multi-label classification, we can use a threshold of 0.7 to determine the predicted labels.
print(classification_report(y_test, y_pred, target_names=labels))

Better Results

Bias against calmness

In [ ]:
y_probs = model.predict(X_test)
plt.hist(y_probs.flatten(), bins=20)
plt.title("Distribution of Probabilities (Model Confidence)")
plt.xlabel("Probability (0 to 1)")
plt.ylabel("Count of Predictions")
plt.show()

Lazy model, no confidence
Model cant see the differmces in Mel Spectorgams

In [ ]:
import pandas as pd
csv_path = os.path.join('..', 'data', 'emotify_lesslables.csv')
df = pd.read_csv(csv_path)
print("Count of samples per class:")
print(df[['energy_joy', 'calm_relax', 'sadness_nostalgia', 'tension_fear']].sum())

Change weights

In [ ]:
import numpy as np


counts = np.array([145, 157, 150, 78])
total = counts.sum()
n_classes = len(counts)

# weights (Inverse Frequency)
weights = total / (n_classes * counts)

class_weights = {i: weights[i] for i in range(n_classes)}

print("Weights that we will give to the model:")
print(f"Energy/Joy: {class_weights[0]:.2f}")
print(f"Calm/Relax: {class_weights[1]:.2f}")
print(f"Sadness: {class_weights[2]:.2f}")
print(f"Tension/Fear: {class_weights[3]:.2f} ")

ADD noise + time shift = 3x Dataset

In [ ]:
import numpy as np

csv_path = os.path.join('..', 'data', 'emotify_final_dataset.csv')
base_folder = os.path.join('..', 'data', 'Emotify_npy') 

def load_dataset_pro(csv_path=csv_path, base_folder=base_folder):
    df = pd.read_csv(csv_path)
    X, Y = [], []
    emotion_cols = ['energy_joy', 'calm_relax', 'sadness_nostalgia', 'tension_fear']
    
    print("Loading and preparing the dataset...")
    
    for i, path in enumerate(df['file_path']):
        full_path = os.path.join(base_folder, path)
        npy_path = full_path.replace('Emotify_Audio', 'Emotify_npy').replace('.mp3', '.npy')
        
        if not os.path.exists(npy_path): continue
            
        data = np.load(npy_path)
        # regualazzation
        data = (data - data.min()) / (data.max() - data.min() + 1e-8)
        
        #prototypo
        X.append(data)
        Y.append(df.iloc[i][emotion_cols].values)
        
        #add noise
        noise = np.random.normal(0, 0.002, data.shape)
        X.append(data + noise)
        Y.append(df.iloc[i][emotion_cols].values)
        
        #time shigfting-
        shift = np.roll(data, shift=np.random.randint(-2, 2), axis=0)
        X.append(shift)
        Y.append(df.iloc[i][emotion_cols].values)

    X = np.array(X)
    X = np.expand_dims(X, axis=-1)
    y = np.array(Y).astype(float)
    
    return X, y, emotion_cols

CRNN trial #3
small learning steps -> stability
early stopping

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models

# CRNN optimization
def build_crnn_model(input_shape=(128, 1292, 1)):
    model = models.Sequential([
       
        layers.Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=input_shape),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)), # (128, 1292) -> (64, 646)
        
        
        layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)), # (64, 646) -> (32, 323)
        
      
        layers.Conv2D(128, (3, 3), activation='relu', padding='same'), # ayjisi σε 128
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)), # (32, 323) -> (16, 161)

        # RESHAPE

        layers.Reshape(target_shape=(161, 16 * 128)), 

        # RNN BLOCK (GRU)
        # 128 units 
        layers.GRU(128, return_sequences=False, dropout=0.2, recurrent_dropout=0.2),
        
        #  DENSE BLOCK
        layers.Dense(64, activation='relu'),
        layers.Dropout(0.4), #  dropout
        
        #output: 4 labels, Sigmoid for multi-label 
        layers.Dense(4, activation='sigmoid') 
    ])
    
    # usage of adam optimizer with a learning rate of 0.0003
    optimizer = tf.keras.optimizers.Adam(learning_rate=0.0003)
    
    model.compile(
        optimizer=optimizer, 
        loss='binary_crossentropy', 
        metrics=['accuracy']
    )
    
    return model

# Early Stopping for preventing overfitting and saving the best model
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss', 
    patience=10, 
    restore_best_weights=True,
    verbose=1
)

# ReduceLROnPlateau
reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss', 
    factor=0.2,   # Reduces the learning rate to 1/5 (e.g., from 0.0003 to 0.00006)
    patience=4,    # Waits for 4 epochs without improvement before acting
    min_lr=1e-6,   # Won't go below this learning rate
    verbose=1
)

In [ ]:
import numpy as np


# energy_joy: 145, calm_relax: 157, sadness_nostalgia: 150, tension_fear: 78
counts = np.array([145, 157, 150, 78])
total = counts.sum()
n_classes = len(counts)

# reverse frequency for class balancing
weights = total / (n_classes * counts)
class_weights = {i: weights[i] for i in range(n_classes)}

print("Βάρη κλάσεων:", class_weights)

In [ ]:
# 1200 samples
X, y, labels = load_dataset_pro(csv_path, base_folder)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 2. learn with the Class Weights
history = model.fit(
    X_train, y_train,
    epochs=50,
    batch_size=32, 
    validation_data=(X_test, y_test),
    class_weight=class_weights,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

In [ ]:
# saving the model in .h5 format

model.save(os.path.join('..', 'model', 'emotify_crnn_pro_final_80.h5'))

# Confirmation
print("The model 'emotify_crnn_pro_final_80.h5' has been saved successfully!")

In [ ]:
from sklearn.metrics import classification_report, multilabel_confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np


y_pred_probs = model.predict(X_test)
y_pred = (y_pred_probs > 0.5).astype(int)

#lassification Report
print("Final results (Epoch 49 Weights):")
print("-" * 60)
print(classification_report(y_test, y_pred, target_names=labels))
print("-" * 60)

#Confusion Matrix for each emotion
mcm = multilabel_confusion_matrix(y_test, y_pred)

plt.figure(figsize=(15, 10))
for i, label in enumerate(labels):
    plt.subplot(2, 2, i+1)
    sns.heatmap(mcm[i], annot=True, fmt='d', cmap='Greens', 
                xticklabels=['Negative', 'Positive'], 
                yticklabels=['Negative', 'Positive'])
    plt.title(f'Confusion Matrix: {label}')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt


plt.style.use('seaborn-v0_8-whitegrid') 
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

#Accuracy
ax1.plot(history.history['accuracy'], label='Training', color='#1f77b4', linewidth=2)
ax1.plot(history.history['val_accuracy'], label='Validation', color='#ff7f0e', linewidth=2)
ax1.set_title('Model Accuracy', fontsize=14, fontweight='bold')
ax1.set_xlabel('Epochs', fontsize=12)
ax1.set_ylabel('Accuracy', fontsize=12)
ax1.legend(loc='lower right', frameon=True)
ax1.grid(True, linestyle='--', alpha=0.7)

#Loss
ax2.plot(history.history['loss'], label='Training', color='#d62728', linewidth=2)
ax2.plot(history.history['val_loss'], label='Validation', color='#2ca02c', linewidth=2)
ax2.set_title('Model Loss', fontsize=14, fontweight='bold')
ax2.set_xlabel('Epochs', fontsize=12)
ax2.set_ylabel('Binary Crossentropy', fontsize=12)
ax2.legend(loc='upper right', frameon=True)
ax2.grid(True, linestyle='--', alpha=0.7)

plt.tight_layout()

plt.savefig(os.path.join('..', 'results', 'training_results_plot.png'), dpi=300)
plt.show()

In [ ]:
from sklearn.metrics import roc_curve, auc
from itertools import cycle


y_score = model.predict(X_test)

#ROC and AUC
fpr = dict()
tpr = dict()
roc_auc = dict()

for i in range(len(labels)):
    fpr[i], tpr[i], _ = roc_curve(y_test[:, i], y_score[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

#micro-average ROC curve
fpr["micro"], tpr["micro"], _ = roc_curve(y_test.ravel(), y_score.ravel())
roc_auc["micro"] = auc(fpr["micro"], tpr["micro"])


plt.figure(figsize=(10, 8))

#Micro-average ROC curve
plt.plot(fpr["micro"], tpr["micro"],
         label=f'Micro-average ROC (area = {roc_auc["micro"]:0.2f})',
         color='deeppink', linestyle=':', linewidth=4)

colors = cycle(['blue', 'green', 'red', 'darkorange'])
for i, color in zip(range(len(labels)), colors):
    plt.plot(fpr[i], tpr[i], color=color, lw=2,
             label=f'ROC curve: {labels[i]} (area = {roc_auc[i]:0.2f})')

#baseline
plt.plot([0, 1], [0, 1], 'k--', lw=2)

plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curves per Class', fontsize=14, fontweight='bold')
plt.legend(loc="lower right", fontsize=10)
plt.grid(True, linestyle='--', alpha=0.6)

plt.savefig('roc_curves_final.png', dpi=300)
plt.show()

τιμές AUC > 0.95 για όλες τις κατηγορίες

In [ ]:
import pandas as pd
history_df = pd.DataFrame(history.history)

history_df.index += 1
history_df.index.name = 'epoch'

history_df.to_csv(os.path.join('..', 'results', 'training_history_log.csv'), index=False)
history_df.to_excel(os.path.join('..', 'results', 'training_history_log.xlsx'), index=False)
print("The training history has been saved to 'training_history_log.csv'!")

In [ ]:
import json
with open(os.path.join('..', 'model', 'labels_map.json'), 'w') as f:
    json.dump(labels, f)

In [ ]:
with open(os.path.join('..', 'results', 'final_classification_report.txt'), 'w', encoding='utf-8') as f:
    f.write("Final Results for CRNN Model\n")
    f.write("-" * 30 + "\n")
    f.write(classification_report(y_test, y_pred, target_names=labels))

print("The report has been saved to 'final_classification_report.txt'!")

In [ ]:
import pandas as pd

y_pred_probs = model.predict(X_test)
y_pred = (y_pred_probs > 0.5).astype(int)


errors_idx = []
for i in range(len(y_test)):
    if not np.array_equal(y_test[i], y_pred[i]):
        errors_idx.append(i)

print(f"Sum of Errors in Test Set: {len(errors_idx)} από {len(y_test)}")



error_details = []

for idx in errors_idx:

    actual_labels = [labels[i] for i, val in enumerate(y_test[idx]) if val == 1]
    predicted_labels = [labels[i] for i, val in enumerate(y_pred[idx]) if val == 1]
    
    error_details.append({
        'File Index': idx,
        'Filename': filenames_test[idx] if 'filenames_test' in locals() else "Unknown",
        'Actual': ", ".join(actual_labels),
        'Predicted': ", ".join(predicted_labels) if predicted_labels else "None"
    })

df_errors = pd.DataFrame(error_details)

print("\n--- List of Incorrect Classifications ---")
display(df_errors.head(20))

df_errors.to_csv(os.path.join('..', 'results', 'model_errors_analysis.csv'), index=False)

genre\feel


In [ ]:
import os
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from tensorflow.keras.models import load_model

BASE_NPY_DIR = os.path.join('..', 'data', 'Emotify_npy')  # Adjust this path as needed
CSV_PATH = os.path.join('..', 'data', 'emotify_lesslabels.csv')  # Adjust this path as needed
MODEL_PATH = os.path.join('..', 'model', 'emotify_crnn_pro_final_80.h5')  # Adjust this path as needed
LABELS = ['energy_joy', 'calm_relax', 'sadness_nostalgia', 'tension_fear']

model = load_model(MODEL_PATH)
df = pd.read_csv(CSV_PATH)
results_list = []

for genre in df['genre'].unique():
    genre_samples = df[df['genre'] == genre]
    X_genre = []
    
    for _, row in genre_samples.iterrows():
        parts = row['file_path'].split('/')
        if len(parts) >= 3:
            relative_path = os.path.join(parts[1], parts[2].replace('.mp3', '.npy'))
            full_path = os.path.join(BASE_NPY_DIR, relative_path)
            
            if os.path.exists(full_path):
                data = np.load(full_path)
                

                d_min, d_max = data.min(), data.max()
                if d_max - d_min != 0:
                    data = (data - d_min) / (d_max - d_min)                
                X_genre.append(data)
    
    if X_genre:
        X_array = np.array(X_genre)
        X_array = X_array.reshape(X_array.shape[0], 128, 1292, 1)
        
        preds = model.predict(X_array, verbose=0)
        mean_scores = np.mean(preds, axis=0) * 100
        
        res = {'Genre': genre}
        for i, label in enumerate(LABELS):
            res[label] = mean_scores[i]
        results_list.append(res)
        print(f"{genre:12}: Calculated {len(X_genre)} files.")

if results_list:
    df_final = pd.DataFrame(results_list).set_index('Genre')
    print("\nREALISTIC PERCENTAGES OF AI PREDICTIONS BY MUSIC GENRE (%)")
    display(df_final.round(2))
    
    plt.figure(figsize=(12, 6))
    sns.heatmap(df_final, annot=True, fmt=".1f", cmap="YlGnBu", cbar_kws={'label': 'Probability %'})
    plt.title("Emotional Profile of Genres", fontsize=14)
    plt.xlabel("Emotional Class")
    plt.ylabel("Music Genre")
    plt.show()